# 🧬 PLM + LLM RAG Pipeline for Kinase Biology Analysis

## What does this pipeline do?

This notebook answers scientific questions about **kinase biology** — for example:
- *"Why does EGFR T790M cause gefitinib resistance?"*
- *"How does BRAF V600E activate the MAPK pathway?"*  
- *"What is the structural basis of ABL1 kinase activation?"*
- *"Why is the DFG motif conserved across kinases?"*

It chains **five stages** together:

| Stage | What happens | Technology used |
|---|---|---|
| **1. Query Orchestration** | LLM extracts the kinase name, mutation, and drug from the question | Groq (Llama-3.3-70B) |
| **2. RAG Retrieval** | Fetches kinase data from UniProt and finds the most relevant literature chunks | FAISS + SentenceTransformers |
| **2b. Sequence Fetch + Mutation** | Fetches wildtype sequence from UniProt, applies the point mutation to generate the mutant | UniProt REST API |
| **3. PLM Mutation Scoring** | Computes ΔLL = LL(mutant) − LL(wildtype) using ESM-2 — how evolutionarily unusual is the mutation? | ESM-2 (Meta) |
| **4. Cross-Modal Verification** | LLM judge integrates summarized literature + ΔLL to explain the mutation's clinical effect | Groq (Llama-3.3-70B) |

### Pipeline Flow
```
Your Kinase Biology Question
     │
     ▼
[Stage 1]  LLM extracts: kinase, mutation (if any), drug (if any)
     │
     ├──────────────────────────┐
     ▼                          ▼
[Stage 2]                  [Stage 2b] (conditional)
Pre-built kinase RAG       Fetch WT sequence + apply mutation
Load FAISS index           Only if mutation present
Top-K literature chunks         │
     │                          │
     │                          │
     └──────────┬───────────────┘
                ▼
         [Stage 3] (conditional)  ESM-2 mutation scoring
         Only if mutation present  ΔLL = logp(mutant) − logp(wildtype)
                ▼
         [Stage 4]  LLM integrates summarized literature + evolutionary evidence → Answer
```


---
## 📦 Step 0: Install Dependencies

Before anything runs, we need to install the Python libraries this pipeline depends on:

- **`groq`** — Python client for the Groq API, which gives us fast access to Llama-3.3-70B
- **`faiss-cpu`** — Facebook's vector similarity search library (the engine behind RAG retrieval)
- **`sentence-transformers`** — Converts text chunks into numerical vectors (embeddings) so FAISS can compare them
- **`torch`** — PyTorch, required by both sentence-transformers and ESM-2
- **`fair-esm`** — Meta's protein language model library (ESM-2), used in Stage 3

> ⚠️ `fair-esm` can take a few minutes to install and will download model weights (~30MB for the small model we use) on first run.

**Google Colab users:** After installing, set your API key like this:
```python
import os; os.environ["GROQ_API_KEY"] = "gsk_..."
```

In [7]:
# Install all required packages — run this cell once before anything else
!pip install groq faiss-cpu sentence-transformers fair-esm transformers pandas tqdm

# Colab users: uncomment the next line and paste your Groq API key
import os; os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"

---
## ⚙️ Step 1: Imports & Configuration

Here we import all libraries and define the global settings that control the pipeline's behavior.

### Key configuration values

- **`GROQ_API_KEY`** — Your API key, loaded from an environment variable (never hard-code secrets!). Get one free at [console.groq.com](https://console.groq.com).
- **`LLM_MODEL`** — The specific LLM we use. `llama-3.3-70b-versatile` is a strong open-source model that's fast on Groq's hardware.
- **`UNIPROT_API`** — The base URL for UniProt's REST API, the world's largest curated protein database.
- **`TOP_K = 5`** — How many text chunks RAG retrieves. Higher = more context for the LLM, but also more tokens and slower responses.

In [8]:
import os
import json
import requests
import numpy as np

# LLM client (Groq hosts Llama models with very fast inference)
from groq import Groq

# Vector similarity search — the core of our RAG retrieval
import faiss

# Converts text into dense embedding vectors for semantic search
from sentence_transformers import SentenceTransformer

# ESM-2 imports
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import pandas as pd
from tqdm import tqdm
from pathlib import Path

# --- Configuration ---

# Load API key from environment variable (set this before running: export GROQ_API_KEY=your_key)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    print("⚠️  GROQ_API_KEY not set — LLM stages will fail, but ESM-2 will still load")

# The Groq-hosted LLM model used in Stage 1 and Stage 4
LLM_MODEL = "llama-3.3-70b-versatile"

# UniProt REST API base URL — our source of protein literature data
UNIPROT_API = "https://rest.uniprot.org/uniprotkb/search"

# How many RAG chunks to retrieve — trade-off between context richness and speed
TOP_K = 5

# ESM-2 configuration
MODEL_NAME = "facebook/esm2_t30_150M_UR50D"
CACHE_DIR = "./cache"
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Imports and config loaded successfully")


# ESM-2 Mutation Scoring Functions

def parse_mutant(mutant):
    """
    Parses a mutant string into its components.

    Args:
        mutant (str): String representing the mutation (e.g., 'A123C').

    Returns:
        tuple: (wild-type amino acid, position index, mutant amino acid).
    """
    wt = mutant[0]
    pos = int(mutant[1:-1]) - 1  # convert 1-indexed biology notation to 0-indexed
    mt = mutant[-1]
    return wt, pos, mt


def build_masked_sequence(sequence_wt, pos):
    """
    Creates a sequence with a <mask> token at the specified position.

    Args:
        sequence_wt (str): The wild-type sequence.
        pos (int): The index to mask.

    Returns:
        str: Sequence with <mask> at pos.
    """
    sequence = list(sequence_wt)
    sequence[pos] = "<mask>"
    return "".join(sequence)


print(f"Using device: {DEVICE}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR).to(DEVICE)
model.eval()

aa_list = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_id = {}
for aa in aa_list:
    tokens = tokenizer.tokenize(aa)
    if len(tokens) != 1:
        raise ValueError(f"Expected single token for {aa}, got {tokens}")
    aa_to_id[aa] = tokenizer.convert_tokens_to_ids(tokens[0])

mask_token_id = tokenizer.mask_token_id
SCORE_CACHE_PATH = Path("mutant_plm_scores.csv")


def score_mutants_with_plm(mutants, sequence_wt, batch_size=32, desc="PLM scoring"):
    """
    Computes zero-shot log-likelihood ratio scores (delta logp) using a Masked LM.

    Args:
        mutants (list): List of mutant strings.
        sequence_wt (str): Wild-type protein sequence.
        batch_size (int): Number of sequences to process per GPU forward pass.
        desc (str): Description for the tqdm progress bar.

    Returns:
        np.ndarray: Array of delta log-probability scores (logp(mt) - logp(wt)).
    """
    if len(mutants) == 0:
        return np.array([], dtype=np.float32)

    MAX_LEN = 1020  # ESM-2 token limit (1022 minus BOS/EOS)
    wt_aas = []
    mt_aas = []
    masked_seqs = []
    for mut in mutants:
        wt, pos, mt = parse_mutant(mut)
        wt_aas.append(wt)
        mt_aas.append(mt)
        # Window sequence around mutation if it exceeds ESM-2 limit
        if len(sequence_wt) > MAX_LEN:
            start = max(0, pos - MAX_LEN // 2)
            end = min(len(sequence_wt), start + MAX_LEN)
            start = max(0, end - MAX_LEN)
            adj_pos = pos - start
            seq_window = sequence_wt[start:end]
        else:
            adj_pos = pos
            seq_window = sequence_wt
        masked_seqs.append(build_masked_sequence(seq_window, adj_pos))

    delta_logp = []
    for start in tqdm(range(0, len(masked_seqs), batch_size), desc=desc):
        batch_seqs = masked_seqs[start : start + batch_size]
        batch_wt = wt_aas[start : start + batch_size]
        batch_mt = mt_aas[start : start + batch_size]

        inputs = tokenizer(
            batch_seqs, return_tensors="pt", padding=True, add_special_tokens=True
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(**inputs).logits

        for idx in range(len(batch_seqs)):
            input_ids = inputs["input_ids"][idx]
            mask_indices = (input_ids == mask_token_id).nonzero(as_tuple=True)[0]
            if len(mask_indices) != 1:
                raise RuntimeError(
                    f"Expected exactly one mask token, found {len(mask_indices)}"
                )

            mask_idx = mask_indices.item()
            log_probs = torch.log_softmax(logits[idx, mask_idx, :], dim=-1)
            wt_id = aa_to_id[batch_wt[idx]]
            mt_id = aa_to_id[batch_mt[idx]]
            delta_logp.append((log_probs[mt_id] - log_probs[wt_id]).item())

    return np.array(delta_logp, dtype=np.float32)


def load_plm_score_cache(cache_path=SCORE_CACHE_PATH):
    """
    Loads cached PLM scores from a CSV file into a dictionary.

    Args:
        cache_path (Path): The file path to the CSV cache.

    Returns:
        dict: A dictionary mapping mutant strings to their float32 PLM scores.
    """
    if not cache_path.exists():
        return {}

    cached_df = pd.read_csv(cache_path)
    required_cols = {"mutant", "plm_delta_logp"}
    if not required_cols.issubset(cached_df.columns):
        raise ValueError(
            f"{cache_path} must contain columns: {sorted(required_cols)}"
        )

    cached_df = (
        cached_df[["mutant", "plm_delta_logp"]]
        .dropna()
        .drop_duplicates(subset="mutant", keep="last")
    )
    return dict(
        zip(cached_df["mutant"].astype(str), cached_df["plm_delta_logp"].astype(np.float32))
    )


def save_plm_score_cache(score_map, cache_path=SCORE_CACHE_PATH):
    """
    Saves a dictionary of PLM scores to a CSV file for future use.

    Args:
        score_map (dict): Dictionary of mutant-to-score mappings.
        cache_path (Path): The file path where the CSV should be saved.
    """
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_df = pd.DataFrame(
        {
            "mutant": list(score_map.keys()),
            "plm_delta_logp": list(score_map.values()),
        }
    ).drop_duplicates(subset="mutant", keep="last")
    cache_df.sort_values("mutant").to_csv(cache_path, index=False)


def get_cached_plm_scores(mutants, sequence_wt, batch_size=32, desc="PLM scoring"):
    """
    Retrieves PLM scores for a list of mutants, computing and caching them if they are missing.

    Args:
        mutants (list): List of mutant strings to score.
        sequence_wt (str): The wild-type protein sequence.
        batch_size (int): Batch size for model inference if scoring is needed.
        desc (str): Description for the progress bar.

    Returns:
        np.ndarray: Array of PLM scores in the same order as the input mutants.
    """
    mutants = [str(mut) for mut in mutants]
    cached_scores = load_plm_score_cache()
    missing_mutants = [mut for mut in mutants if mut not in cached_scores]

    if missing_mutants:
        missing_scores = score_mutants_with_plm(
            missing_mutants,
            sequence_wt,
            batch_size=batch_size,
            desc=desc,
        )
        cached_scores.update(
            {mut: float(score) for mut, score in zip(missing_mutants, missing_scores)}
        )
        save_plm_score_cache(cached_scores)

    return np.array([cached_scores[mut] for mut in mutants], dtype=np.float32)

✅ Imports and config loaded successfully
Using device: cuda


Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


---
## 🧠 Stage 1: Query Orchestration (LLM)

### What this function does
We send the user's question to an LLM that extracts a structured JSON object containing:

1. **`kinase`** — Gene name of the kinase (e.g. `"EGFR"`)
2. **`mutation`** — Point mutation in standard notation (e.g. `"T790M"`)
3. **`drug`** — Relevant drug if mentioned (e.g. `"gefitinib"`)
4. **`pubmed_query`** — Optimized literature search string

In [9]:
def decompose_query(user_question: str) -> dict:
    """
    Stage 1: Query Orchestration.
    Extracts kinase name, mutation, drug from the user's question.
    """
    client = Groq(api_key=GROQ_API_KEY)

    system_prompt = (
        "You are a computational biology expert specializing in kinase biology and cancer mutations. "
        "Given a user question, extract structured information and respond ONLY with a JSON object "
        "with keys: kinase, mutation, drug. "
        "kinase: gene name of the kinase (e.g. EGFR, BRAF, ABL1). "
        "mutation: point mutation in standard notation (e.g. T790M, V600E) or null if not mentioned. "
        "drug: drug name if mentioned (e.g. gefitinib, imatinib) or null if not mentioned. "
    )

    response = client.chat.completions.create(
        model=LLM_MODEL,
        max_tokens=512,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print("[Stage 1] ⚠️  JSON parsing failed — using fallback")
        return {
            "kinase": "EGFR",
            "mutation": None,
            "drug": None,
        }

### 🧪 Test Stage 1

In [10]:
test_question = "Why does EGFR T790M cause resistance to gefitinib?"

query_info = decompose_query(test_question)
print(json.dumps(query_info, indent=2))

kinase   = query_info.get("kinase", "EGFR")
mutation = query_info.get("mutation")
drug     = query_info.get("drug")
print(f"\n🔬 Kinase: {kinase} | Mutation: {mutation} | Drug: {drug}")

{
  "kinase": "EGFR",
  "mutation": "T790M",
  "drug": "gefitinib"
}

🔬 Kinase: EGFR | Mutation: T790M | Drug: gefitinib


---
## 📚 Stage 2: RAG — Build Index & Retrieve

Fetches kinase literature from UniProt using the kinase gene name, builds a FAISS index, and retrieves the most relevant chunks for the question.

The retrieved chunks provide literature context about the kinase's function, active site, inhibitor binding, and known mutations — this feeds directly into Stage 4's LLM judge.

In [11]:
class KinaseChunksRAG:
    """
    Pre-built RAG index for kinase data from UniProt chunks.
    Loads FAISS index and chunk metadata from files.
    """

    def __init__(self, index_path: str = "kinase_faiss.index", chunks_path: str = "kinase_chunks_indexed.json", embedder_name: str = "all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(embedder_name)
        self.index = faiss.read_index(index_path)
        with open(chunks_path, 'r') as f:
            self.chunks = json.load(f)
        print(f"[KinaseRAG] Loaded FAISS index with {self.index.ntotal} vectors and {len(self.chunks)} chunks")

    def retrieve(self, question: str, kinase: str | None = None, top_k: int = TOP_K) -> list[str]:
        """Retrieves top-k relevant chunks, prioritizing those matching the kinase gene name."""
        q_emb = self.embedder.encode([question], show_progress_bar=False)
        q_emb = np.array(q_emb, dtype="float32")
        _, indices = self.index.search(q_emb, top_k * 3)
        candidates = [self.chunks[i] for i in indices[0] if i < len(self.chunks)]

        if kinase:
            kinase_upper = kinase.upper()
            matched = [c for c in candidates if kinase_upper in c.get('gene_name', '').upper()]
            others  = [c for c in candidates if kinase_upper not in c.get('gene_name', '').upper()]
            candidates = (matched + others)[:top_k]
        else:
            candidates = candidates[:top_k]

        return [c['text'] for c in candidates]


### 🧪 Test Stage 2

In [12]:
kinase = query_info.get("kinase", "EGFR")

# Use pre-built kinase chunks index instead of dynamic UniProt fetch
rag = KinaseChunksRAG()

chunks = rag.retrieve(test_question, kinase=kinase, top_k=TOP_K)
print(f"\n📄 Top {len(chunks)} retrieved chunks:")
for i, chunk in enumerate(chunks, 1):
    print(f"\n  [{i}] {chunk[:150]}..." if len(chunk) > 150 else f"\n  [{i}] {chunk}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[KinaseRAG] Loaded FAISS index with 3691 vectors and 3691 chunks

📄 Top 5 retrieved chunks:

  [1] FUNCTION INFORMATION
UniProt accession: P00533
Entry name: EGFR_HUMAN
Protein name: Epidermal growth factor receptor (EC 2.7.10.1) (Proto-oncogene c-E...

  [2] SUMMARY INFORMATION
UniProt accession: P00533
Entry name: EGFR_HUMAN
Protein name: Epidermal growth factor receptor (EC 2.7.10.1) (Proto-oncogene c-Er...

  [3] PATHWAY AND ACTIVITY REGULATION INFORMATION
UniProt accession: P00533
Entry name: EGFR_HUMAN
Protein name: Epidermal growth factor receptor (EC 2.7.10...

  [4] BINDING SITE INFORMATION
UniProt accession: P00533
Entry name: EGFR_HUMAN
Protein name: Epidermal growth factor receptor (EC 2.7.10.1) (Proto-oncogene...

  [5] FUNCTION INFORMATION
UniProt accession: P17948
Entry name: VGFR1_HUMAN
Protein name: Vascular endothelial growth factor receptor 1 (VEGFR-1) (EC 2.7.1...


---
## 🧬 Stage 2b: Fetch Wildtype Sequence + Apply Mutation

### What this stage does
1. Fetches the **canonical wildtype sequence** from UniProt (Swiss-Prot reviewed)
2. Parses the mutation string (e.g. `T790M` → position 790, from T to M)
3. Applies the point mutation to generate the **mutant sequence**

Both sequences are passed to Stage 3 for ESM-2 scoring.

### Mutation notation
Standard single-letter amino acid notation:
- `T790M` → position 790, Threonine (T) → Methionine (M)
- `V600E` → position 600, Valine (V) → Glutamate (E)

The function validates that the wildtype residue at the specified position matches the notation before applying the mutation.

In [13]:
import re

def fetch_wt_sequence(gene_name: str) -> tuple[str, str]:
    """Fetches canonical wildtype sequence from UniProt Swiss-Prot."""
    params = {
        "query": f"gene_exact:{gene_name} AND organism_id:9606 AND reviewed:true",
        "format": "json",
        "size": 1,
        "fields": "accession,sequence",
    }
    resp = requests.get(UNIPROT_API, params=params, timeout=30)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    if not results:
        raise ValueError(f"No reviewed UniProt entry found for {gene_name}")
    acc = results[0].get("primaryAccession", gene_name)
    seq = results[0].get("sequence", {}).get("value", "")
    print(f"[SEQ] {gene_name} → {acc} ({len(seq)} aa)")
    return acc, seq


def apply_mutation(seq: str, mutation: str) -> tuple[int, str, str, str]:
    """
    Parses and applies a point mutation to a sequence.
    mutation format: e.g. 'T790M' (WT_residue + position + MUT_residue)
    Returns: (position, wt_residue, mut_residue, mutant_sequence)
    """
    match = re.match(r"([A-Z])(\d+)([A-Z])", mutation.upper())
    if not match:
        raise ValueError(f"Cannot parse mutation: {mutation}")

    wt_aa, pos_str, mut_aa = match.groups()
    pos = int(pos_str)  # 1-indexed

    if pos > len(seq):
        raise ValueError(f"Position {pos} out of range for sequence of length {len(seq)}")

    actual_wt = seq[pos - 1]  # convert to 0-indexed
    if actual_wt != wt_aa:
        print(f"[MUT] ⚠️  Position {pos}: expected {wt_aa}, found {actual_wt} — applying anyway")

    mut_seq = seq[:pos - 1] + mut_aa + seq[pos:]
    print(f"[MUT] Applied {mutation}: position {pos} {actual_wt} → {mut_aa}")
    return pos, actual_wt, mut_aa, mut_seq

### 🧪 Test Stage 2b

In [14]:
acc, wt_seq = fetch_wt_sequence(kinase)

mut_pos = mut_aa_wt = mut_aa_mut = None
mut_seq = None

if mutation:
    mut_pos, mut_aa_wt, mut_aa_mut, mut_seq = apply_mutation(wt_seq, mutation)
    print(f"\n✅ WT sequence:  {wt_seq[mut_pos-3:mut_pos+2]}  (pos {mut_pos})")
    print(f"   Mut sequence: {mut_seq[mut_pos-3:mut_pos+2]}  (pos {mut_pos})")
else:
    print("⚠️  No mutation specified — Stage 3 will score WT sequence only")

[SEQ] EGFR → P00533 (1210 aa)
[MUT] Applied T790M: position 790 T → M

✅ WT sequence:  LITQL  (pos 790)
   Mut sequence: LIMQL  (pos 790)


---
## 🔬 Stage 3: PLM Mutation Scoring (ESM-2 ΔLL)

### What is ΔLL?
ESM-2 assigns a **log-probability score** to each amino acid at each position based on evolutionary patterns learned from millions of protein sequences. We compute:

```
ΔLL = logp(mutant) − logp(wildtype)
```

Where logp is the log-probability of the amino acid at the mutation position when that position is masked and predicted by the model.

| ΔLL | Interpretation |
|-----|---------------|
| Near 0 | Mutation is evolutionarily neutral — both amino acids are equally likely |
| Negative | Mutation is evolutionarily unusual — the wildtype amino acid is preferred, mutation likely disruptive |
| Strongly negative (< −1) | Position is highly conserved — mutation almost certainly affects function |

### Why this is meaningful for kinases
Kinase active site residues (DFG motif, gatekeeper, P-loop) are under strong evolutionary pressure. A mutation at these positions typically has a very negative ΔLL — ESM-2 independently confirms what the literature says about functional importance.

### Masked position scoring
We mask the mutation position in the wildtype sequence and compute the log-probability of both the wildtype and mutant amino acids at that position:

```python
masked_seq = wt_seq[:pos] + "<mask>" + wt_seq[pos+1:]
logp_wt = model.logp(masked_seq, wt_aa)
logp_mut = model.logp(masked_seq, mut_aa)
delta_ll = logp_mut - logp_wt
```

This gives a direct, position-specific signal for mutation effect.

In [15]:
def score_mutation_esm(
    acc: str,
    wt_seq: str,
    mut_seq: str,
    mut_position: int,
    mutation: str,
) -> dict:
    """
    Stage 3: ESM-2 Mutation Scoring using Masked LM.

    Computes ΔLL = logp(mutant) − logp(wildtype) at the masked mutation position.
    A negative ΔLL means the mutation is evolutionarily unusual.

    Returns a dict with:
      - delta_ll: logp(mt) - logp(wt) (negative = disruptive)
      - accession, mutation, position
    """
    print(f"[PLM] Scoring mutation {mutation} on {acc} using ESM-2 Masked LM...")

    scores = get_cached_plm_scores([mutation], wt_seq, desc="PLM scoring")
    delta_ll = scores[0]

    result = {
        "accession": acc,
        "mutation": mutation,
        "position": mut_position,
        "delta_ll": delta_ll,
        # Not available in masked LM approach
        "ll_wt": None,
        "ll_mut": None,
        "ll_wt_global": None,
        "ll_mut_global": None,
    }

    print(f"[PLM] ΔLL = {delta_ll:.4f}  ({'disruptive ⚠️' if delta_ll < -0.5 else 'neutral'})")
    return result

### 🧪 Test Stage 3

In [16]:
if mut_seq and mut_pos:
    plm_result = score_mutation_esm(acc, wt_seq, mut_seq, mut_pos, mutation)
else:
    print("⚠️  No mutation to score — skipping Stage 3")
    plm_result = {}

[PLM] Scoring mutation T790M on P00533 using ESM-2 Masked LM...


PLM scoring: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]

[PLM] ΔLL = -3.4545  (disruptive ⚠️)


---
## ⚖️ Stage 4: Cross-Modal Verification (LLM Judge)

### The core idea
Two independent evidence streams are compared:
- **RAG literature evidence** — top-K chunks retrieved from UniProt kinase index
- **PLM mutation evidence** — ESM-2 ΔLL score at the mutation position

### Interpreting ΔLL in the prompt
The LLM is instructed to interpret the ΔLL signal alongside the literature summary:

| ΔLL | Literature says disruptive | Interpretation |
|-----|--------------------------|---------------|
| Strongly negative | Yes | High-confidence: mutation disrupts conserved position |
| Strongly negative | No | Flag contradiction: evolutionarily unusual but literature claims benign |
| Near 0 | Yes | Flag contradiction: literature claims disruption but position isn't conserved |
| Near 0 | No | Consistent: neutral mutation, no functional effect expected |

In [17]:
def cross_modal_verify(
    user_question: str,
    plm_result: dict,
    mutation: str | None,
    drug: str | None,
) -> str:
    """
    Stage 4: Cross-Modal Verification for kinase biology questions.
    Integrates summarized literature evidence with ESM-2 ΔLL mutation scoring.
    """
    client = Groq(api_key=GROQ_API_KEY)


    if plm_result:
        plm_summary = (
            f"  Mutation: {plm_result.get('mutation')} at position {plm_result.get('position')}\n"
            f"  ΔLL = {plm_result.get('delta_ll', 0):.4f} "
            f"({'strongly disruptive' if plm_result.get('delta_ll', 0) < -1 else 'moderately disruptive' if plm_result.get('delta_ll', 0) < -0.5 else 'neutral'})"
        )
    else:
        plm_summary = "  No PLM results available."

    drug_context = f"The drug in question is {drug}." if drug else ""

    prompt = f"""You are a computational biology expert specializing in kinase biology.
A user asked: \"{user_question}\"
{drug_context}

ESM-2 MUTATION EVIDENCE (ΔLL = logp(mutant) − logp(wildtype) at the masked mutation position):
{plm_summary}

Interpretation guide for ΔLL:
- Near 0: position is not evolutionarily conserved — mutation is neutral
- Negative (< −0.5): position is conserved — mutation disrupts a functionally important residue
- Strongly negative (< −1): position is highly conserved — mutation almost certainly affects function

Task:
1. Answer the user's question using the literature evidence.
2. If mutation data is available, interpret the ΔLL score: does it confirm that this position is evolutionarily conserved?
3. Integrate structural, functional, and evolutionary evidence for a comprehensive explanation.
4. Be concise and scientifically rigorous."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()

### 🧪 Test Stage 4

In [18]:
final_answer = cross_modal_verify(
    test_question, plm_result,
    mutation=mutation, drug=drug,
)

print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(final_answer)

FINAL ANSWER
The EGFR T790M mutation causes resistance to gefitinib by altering the ATP-binding pocket of the enzyme, reducing the drug's binding affinity. Literature evidence suggests that the T790M mutation introduces a bulky methionine residue, which sterically hinders gefitinib binding (1). 

The ΔLL score of -3.4545 indicates that the T790M mutation is strongly disruptive, confirming that this position is highly evolutionarily conserved. This suggests that the threonine residue at position 790 plays a crucial role in the function of EGFR, and its substitution with methionine significantly affects the protein's function.

Structurally, the T790M mutation is located in the kinase domain of EGFR, near the gefitinib binding site. Functionally, this mutation reduces the inhibitory efficacy of gefitinib, leading to continued signaling and resistance to the drug. Evolutionarily, the high conservation of this residue highlights its importance in maintaining the protein's function.

In sum

---
## 🚀 Full Pipeline: `run_pipeline()`

Chains all stages together:

- Stage 1 → extracts `kinase`, `mutation`, `drug`
- Stage 2 → RAG retrieval from pre-built kinase chunks index
- Stage 2b → fetches WT sequence, applies mutation
- Stage 3 → ESM-2 ΔLL at mutation position
- Stage 4 → LLM integrates summarized literature + ΔLL

In [23]:
def run_pipeline(user_question: str) -> str:
    """
    Kinase biology analysis pipeline:
    1. LLM extracts kinase, mutation, and drug
    2. RAG retrieval from pre-built kinase chunks index
    2b. Fetch WT sequence + apply mutation (if applicable)
    3. ESM-2 ΔLL scoring (if mutation present)
    4. Cross-modal verification integrating summarized literature + evolutionary evidence
    """
    print("\n" + "=" * 60)
    print(f"USER QUESTION: {user_question}")
    print("=" * 60)

    # Stage 1
    print("\n[Stage 1] Decomposing query with LLM...")
    query_info = decompose_query(user_question)
    kinase   = query_info.get("kinase") or None
    mutation = query_info.get("mutation")
    drug     = query_info.get("drug")
    print(f"  Kinase   : {kinase}")
    print(f"  Mutation : {mutation}")
    print(f"  Drug     : {drug}")

    # Stage 2
    print("\n[Stage 2] RAG retrieval from kinase chunks...")
    rag = KinaseChunksRAG()
    chunks = rag.retrieve(user_question, kinase=kinase, top_k=TOP_K)
    print(f"  Retrieved {len(chunks)} chunks.")

    # Stage 2b
    print("\n[Stage 2b] Fetching WT sequence and applying mutation...")
    acc, wt_seq = None, None
    mut_pos = None
    mut_seq = None
    if kinase:
        acc, wt_seq = fetch_wt_sequence(kinase)
        if mutation:
            mut_pos, _, _, mut_seq = apply_mutation(wt_seq, mutation)
    else:
        print("  No specific kinase identified — skipping sequence fetch.")

    # Stage 3
    print("\n[Stage 3] Running ESM-2 mutation scoring...")
    plm_result = {}
    if mut_seq and mut_pos:
        plm_result = score_mutation_esm(acc, wt_seq, mut_seq, mut_pos, mutation)
    else:
        print("  No mutation — skipping PLM scoring.")

    # Stage 4
    print("\n[Stage 4] Cross-modal verification...")
    final_answer = cross_modal_verify(user_question, plm_result, mutation, drug)

    print("\n" + "=" * 60)
    print("FINAL ANSWER:")
    print("=" * 60)
    print(final_answer)
    return final_answer

---
## ▶️ Run It!

Run the full pipeline — or try your own PPI question.

In [24]:
# Try different kinase resistance questions:
#   "Why does EGFR T790M cause gefitinib resistance?"
#   "How does BRAF V600E drive melanoma?"
#   "Why does BCR-ABL T315I resist imatinib?"

example_question = "Why does EGFR T790M cause resistance to gefitinib?"

answer = run_pipeline(example_question)


USER QUESTION: Why does EGFR T790M cause resistance to gefitinib?

[Stage 1] Decomposing query with LLM...
  Kinase   : EGFR
  Mutation : T790M
  Drug     : gefitinib

[Stage 2] RAG retrieval from kinase chunks...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[KinaseRAG] Loaded FAISS index with 3691 vectors and 3691 chunks
  Retrieved 5 chunks.

[Stage 2b] Fetching WT sequence and applying mutation...
[SEQ] EGFR → P00533 (1210 aa)
[MUT] Applied T790M: position 790 T → M

[Stage 3] Running ESM-2 mutation scoring...
[PLM] Scoring mutation T790M on P00533 using ESM-2 Masked LM...
[PLM] ΔLL = -3.4545  (disruptive ⚠️)

[Stage 4] Cross-modal verification...

FINAL ANSWER:
The EGFR T790M mutation causes resistance to gefitinib due to its location in the ATP-binding pocket of the EGFR tyrosine kinase domain. The threonine to methionine substitution at position 790 introduces a bulkier side chain, which sterically hinders gefitinib binding, thereby reducing its inhibitory efficacy (1).

The ΔLL score of -3.4545 for the T790M mutation indicates that this position is highly conserved, with the mutation being strongly disruptive to the function. This score confirms that the position is evolutionarily conserved, suggesting that the mutation likely affec

---
## 🧪 Evaluation: Kinase QA Gold Set + LLM-as-Judge

Evaluates the full pipeline on a curated set of kinase biology questions with known reference answers.

### Method
- **Gold set**: 15 questions covering all 8 pipeline categories (resistance, activation, structural, etc.)
- **Scorer**: LLM-as-judge rates each pipeline answer 1–5 against the reference answer
- **Metrics**: mean score, failure analysis

### Score rubric
| Score | Meaning |
|---|---|
| 5 | Correct, complete, no hallucination |
| 4 | Mostly correct, minor omissions |
| 3 | Partially correct, key facts present but incomplete |
| 2 | Mostly wrong or significant hallucination |
| 1 | Completely wrong or off-topic |


In [25]:
# --- Kinase QA Gold Set ---
KINASE_GOLD_SET = [
    # resistance_mutation
    {
        "question": "Why does EGFR T790M cause resistance to gefitinib?",
        "reference": "T790M substitutes the gatekeeper threonine at position 790 with methionine, sterically blocking gefitinib binding at the ATP pocket while maintaining kinase activity. The bulkier methionine side chain reduces inhibitor affinity ~1000-fold. Additionally, T790M restores the affinity for ATP, outcompeting gefitinib."
    },
    {
        "question": "Why does BCR-ABL T315I cause resistance to imatinib?",
        "reference": "T315 is the gatekeeper residue of ABL1. The T315I mutation replaces threonine with isoleucine, eliminating a critical hydrogen bond with imatinib and introducing steric clash that prevents imatinib binding. This position is required for imatinib contact and its loss abrogates drug efficacy."
    },
    {
        "question": "How does BRAF V600E cause constitutive kinase activation?",
        "reference": "V600 is located in the activation loop of BRAF. The V600E substitution mimics phosphorylation of the activation loop, shifting BRAF into a constitutively active conformation without requiring upstream RAS signaling. This leads to sustained MEK/ERK pathway activation driving melanoma."
    },
    # activation_mechanism
    {
        "question": "How does EGFR become activated upon EGF binding?",
        "reference": "EGF binding to the extracellular domain induces receptor dimerization. Dimerization drives asymmetric kinase domain interaction where one kinase domain allosterically activates the other. This leads to trans-autophosphorylation of tyrosine residues in the activation loop, stabilizing the active conformation."
    },
    {
        "question": "What is the role of the DFG motif in kinase activation?",
        "reference": "The DFG motif (Asp-Phe-Gly) at the start of the activation loop is a conserved switch. In the DFG-in conformation, the aspartate coordinates Mg2+ for ATP binding and the kinase is active. In the DFG-out conformation, the phenylalanine flips into the ATP pocket, inactivating the kinase. Many inhibitors like imatinib bind the DFG-out state."
    },
    # structural_mechanism
    {
        "question": "What is the structural basis of ATP binding in kinases?",
        "reference": "ATP binds in the cleft between the N-lobe and C-lobe of the kinase domain. The adenine ring forms hydrogen bonds with the hinge region. The phosphate groups are coordinated by the glycine-rich P-loop (GXGXXG) and the DFG aspartate via Mg2+. The gatekeeper residue controls access to a hydrophobic back pocket adjacent to the ATP site."
    },
    {
        "question": "Why is the gatekeeper residue important for kinase inhibitor selectivity?",
        "reference": "The gatekeeper residue sits at the entrance to a hydrophobic back pocket adjacent to the ATP binding site. Its size and chemistry determine whether bulky inhibitors can access this pocket. A small threonine (as in EGFR) allows access; a larger residue blocks it. This makes the gatekeeper a key determinant of inhibitor selectivity and a frequent site of resistance mutations."
    },
    # inhibitor_selectivity
    {
        "question": "Why is imatinib selective for BCR-ABL over most other kinases?",
        "reference": "Imatinib binds the inactive DFG-out conformation of ABL1. Most kinases cannot adopt this conformation readily, conferring selectivity. The drug also makes specific contacts with ABL1 residues in the P-loop and hinge that are not conserved across the kinome. Additionally, the small threonine gatekeeper (T315) in ABL1 allows imatinib access to the back pocket."
    },
    # regulatory_mechanism
    {
        "question": "How does Src kinase autoinhibition work?",
        "reference": "Src is autoinhibited by intramolecular interactions. Phosphorylation of Y527 (in the C-terminal tail) by CSK allows the SH2 domain to bind the tail, and the SH3 domain binds the linker between SH2 and kinase. These interactions stabilize the inactive conformation. Dephosphorylation of Y527 or SH2/SH3 displacement by competing ligands releases autoinhibition."
    },
    # substrate_specificity
    {
        "question": "What determines the substrate specificity of kinases?",
        "reference": "Kinase substrate specificity is determined by the sequence context surrounding the phosphorylation site (consensus motif), docking sites distant from the phosphorylation site, subcellular localization, and scaffolding proteins. The activation loop and substrate binding groove shape the consensus motif preference. For example, PKA prefers R-R-x-S/T while CDKs prefer S/T-P-x-K/R."
    },
    # disease_association
    {
        "question": "How does ALK rearrangement drive lung cancer?",
        "reference": "ALK gene rearrangements (most commonly EML4-ALK fusion) create a constitutively active kinase by fusing the ALK kinase domain to a dimerization domain from EML4. The fusion protein is constitutively dimerized and active, driving downstream RAS/MAPK, PI3K/AKT, and JAK/STAT signaling that promotes proliferation and survival in non-small cell lung cancer."
    },
    {
        "question": "Why does EGFR amplification contribute to glioblastoma?",
        "reference": "EGFR amplification in glioblastoma leads to receptor overexpression, increasing ligand-independent signaling and receptor dimerization probability. A common variant (EGFRvIII) deletes exons 2-7, creating a constitutively active receptor that continuously activates PI3K/AKT and RAS/MAPK pathways, driving tumor growth and resistance to apoptosis."
    },
    # evolutionary_conservation
    {
        "question": "Why is the DFG motif conserved across kinases?",
        "reference": "The DFG motif is conserved because it performs an essential catalytic function: the aspartate coordinates Mg2+-ATP for phosphotransfer, the phenylalanine stabilizes the hydrophobic spine controlling active/inactive transitions, and the glycine provides conformational flexibility needed for the DFG-flip. Mutations at these positions typically abolish catalytic activity, creating strong negative selection pressure."
    },
    {
        "question": "Why is the lysine in the VAIK motif conserved in kinases?",
        "reference": "The lysine in the VAIK motif (beta3 strand) forms a salt bridge with the glutamate of the alphaC helix and coordinates the alpha and beta phosphates of ATP. This interaction is essential for proper ATP positioning for catalysis. Mutation of this lysine (e.g., K→A or K→M) abolishes kinase activity, hence it is invariant across the kinome."
    },
    {
        "question": "How does PKA phosphorylation of its activation loop regulate its activity?",
        "reference": "PKA requires phosphorylation of T197 in its activation loop for full activity. Phospho-T197 forms electrostatic interactions that stabilize the active conformation of the catalytic loop and properly position the substrate binding site. Without this phosphorylation the activation loop is disordered and substrate binding is impaired. PDK1 phosphorylates this site in vivo."
    },
]


def llm_judge_score(question: str, reference: str, pipeline_answer: str) -> dict:
    """LLM-as-judge: scores pipeline answer 1-5 against reference answer."""
    client = Groq(api_key=GROQ_API_KEY)

    prompt = f"""You are an expert in kinase biology evaluating an AI system's answer.

Question: {question}

Reference answer (ground truth):
{reference}

Pipeline answer (to evaluate):
{pipeline_answer}

Score the pipeline answer on a scale of 1-5:
5 = Correct and complete, all key facts present, no hallucination
4 = Mostly correct, minor omissions or imprecision
3 = Partially correct, some key facts present but important gaps
2 = Mostly wrong or contains significant hallucination
1 = Completely wrong, off-topic, or refuses to answer

Respond with JSON only: {{"score": <1-5>, "reason": "<one sentence>"}}"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"score": 0, "reason": f"parse error: {raw[:80]}"}


print(f"Gold set loaded: {len(KINASE_GOLD_SET)} questions")


Gold set loaded: 15 questions


In [26]:
import time

def run_evaluation(gold_set: list[dict], delay_sec: float = 2.0) -> list[dict]:
    """
    Runs the full pipeline on every question in the gold set,
    scores each answer with LLM-as-judge, and returns results.
    """
    results = []
    for idx, item in enumerate(gold_set):
        q   = item["question"]
        ref = item["reference"]
        print(f"\n[{idx+1}/{len(gold_set)}] {q[:80]}..." if len(q) > 80 else f"\n[{idx+1}/{len(gold_set)}] {q}")

        try:
            answer = run_pipeline(q)
            judgment = llm_judge_score(q, ref, answer)
            score = judgment.get("score", 0)
            reason = judgment.get("reason", "")
        except Exception as e:
            answer = ""
            score = 0
            reason = f"ERROR: {e}"

        print(f"  Score: {score}/5 — {reason}")
        results.append({
            "question": q,
            "reference": ref,
            "pipeline_answer": answer,
            "score": score,
            "reason": reason,
        })

        if delay_sec > 0:
            time.sleep(delay_sec)  # avoid rate limiting

    return results


def print_eval_summary(results: list[dict]):
    """Prints score summary."""
    valid = [r for r in results if r["score"] > 0]
    mean_score = sum(r["score"] for r in valid) / len(valid) if valid else 0

    print("\n" + "=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Overall mean score : {mean_score:.2f} / 5  ({len(valid)}/{len(results)} completed)")
    print(f"Accuracy (score≥4) : {sum(1 for r in valid if r['score'] >= 4)}/{len(valid)} = {sum(1 for r in valid if r['score'] >= 4)/len(valid)*100:.1f}%")


    print("\nFailed / low-scoring (score ≤ 2):")
    low = [r for r in valid if r["score"] <= 2]
    if not low:
        print("  None")
    for r in low:
        print(f"  [{r['score']}] {r['question'][:70]}")
        print(f"       {r['reason']}")


# Run it
eval_results = run_evaluation(KINASE_GOLD_SET, delay_sec=2.0)
print_eval_summary(eval_results)

# Save results to JSON for later inspection
with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("\nResults saved to eval_results.json")



[1/15] Why does EGFR T790M cause resistance to gefitinib?

USER QUESTION: Why does EGFR T790M cause resistance to gefitinib?

[Stage 1] Decomposing query with LLM...
  Kinase   : EGFR
  Mutation : T790M
  Drug     : gefitinib

[Stage 2] RAG retrieval from kinase chunks...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[KinaseRAG] Loaded FAISS index with 3691 vectors and 3691 chunks
  Retrieved 5 chunks.

[Stage 2b] Fetching WT sequence and applying mutation...
[SEQ] EGFR → P00533 (1210 aa)
[MUT] Applied T790M: position 790 T → M

[Stage 3] Running ESM-2 mutation scoring...
[PLM] Scoring mutation T790M on P00533 using ESM-2 Masked LM...
[PLM] ΔLL = -3.4545  (disruptive ⚠️)

[Stage 4] Cross-modal verification...

FINAL ANSWER:
The EGFR T790M mutation causes resistance to gefitinib due to its location in the ATP-binding pocket of the enzyme. Gefitinib is a tyrosine kinase inhibitor that binds to the active site of EGFR, blocking ATP binding and thereby inhibiting kinase activity. The T790M mutation introduces a bulky methionine residue, which sterically hinders gefitinib binding, reducing its affinity for the enzyme (1).

The ΔLL score of -3.4545 for the T790M mutation indicates that this position is highly conserved, suggesting that the mutation disrupts a functionally important residue. This is consi

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[KinaseRAG] Loaded FAISS index with 3691 vectors and 3691 chunks
  Retrieved 5 chunks.

[Stage 2b] Fetching WT sequence and applying mutation...
  No specific kinase identified — skipping sequence fetch.

[Stage 3] Running ESM-2 mutation scoring...
  No mutation — skipping PLM scoring.

[Stage 4] Cross-modal verification...

FINAL ANSWER:
The lysine in the VAIK motif is conserved in kinases due to its crucial role in ATP binding and catalysis. Literature evidence suggests that this lysine residue plays a pivotal role in stabilizing the ATP molecule and facilitating the phosphate transfer reaction (Taylor and Kornev, 2011). 

The lack of available ESM-2 mutation evidence (ΔLL score) precludes direct interpretation of the evolutionary conservation at this specific position. However, based on the functional importance of the lysine in the VAIK motif, it is reasonable to infer that this residue is conserved across kinases.

Structurally, the VAIK motif is situated in the kinase active site

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[KinaseRAG] Loaded FAISS index with 3691 vectors and 3691 chunks
  Retrieved 5 chunks.

[Stage 2b] Fetching WT sequence and applying mutation...
[SEQ] PRKACA → P17612 (351 aa)

[Stage 3] Running ESM-2 mutation scoring...
  No mutation — skipping PLM scoring.

[Stage 4] Cross-modal verification...

FINAL ANSWER:
PKA (Protein Kinase A) is a crucial kinase that regulates various cellular processes. Phosphorylation of its activation loop is a key regulatory mechanism. The activation loop, also known as the T-loop, is a conserved region in kinases that controls substrate binding and catalytic activity.

Upon phosphorylation, the activation loop undergoes a conformational change, allowing PKA to adopt an active conformation. This phosphorylation event is mediated by another kinase, PDK1 (Phosphoinositide-Dependent Kinase 1), which specifically targets the activating phosphorylation site (Thr197 in PKA). The phosphorylated activation loop then stabilizes the active conformation of PKA, enabli